# Notebook 36 — Manuscript tables and figures

This notebook aggregates completed upstream JSON/CSV/history/result artifacts only. It does not load a model, construct a dataset or loader, run inference, train, or recompute scientific results. Missing results remain explicitly `MISSING`; no value is synthesized.

In [1]:
from pathlib import Path
import json
import pandas as pd
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks": PROJECT_ROOT = PROJECT_ROOT.parent
def load_json_or_missing(relative_path):
    path = PROJECT_ROOT / relative_path
    return json.loads(path.read_text()) if path.is_file() else {"status": "MISSING", "path": str(relative_path)}
evidence_paths = {
    "pretraining_backbone_provenance": "reports/backbone_transfer_v1_3.json",
    "downstream_convergence": "reports/MEPI_V1_4_BASELINE_VALIDATION_METRICS.json",
    "loss_weight_sensitivity": "experiments/finetune_v1_5_xlstm_depth8/loss_weight_sensitivity/comparison.json",
    "final_test_metrics": "reports/MEPI_V1_5_FINAL_TEST_RESULTS.json",
    "computational_profile": "reports/benchmark_pretraining_comprehensive.json",
    "frequency_screening": "reports/MEPI_V1_5_FREQUENCY_SCREENING.json",
}
evidence = {section: load_json_or_missing(path) for section, path in evidence_paths.items()}
final_test = evidence["final_test_metrics"]
screening = evidence["frequency_screening"]
required_final_metric_keys = {
    "TEST_EFFICIENCY_MAE", "TEST_EFFICIENCY_RMSE", "TEST_EFFICIENCY_R2",
    "TEST_PLOSS_MAE", "TEST_PLOSS_RMSE", "TEST_PLOSS_R2",
    "TEST_LSP_MAE", "TEST_LSP_RMSE", "TEST_LSP_R2", "TEST_LSP_MAPE_PERCENT",
}
final_test_available = (
    final_test.get("EVALUATION_STATUS") == "FINAL_TEST_COMPLETE"
    and final_test.get("TEST_ACCESSED") is True
    and final_test.get("TEST_EVALUATION_COUNT") == 1
    and required_final_metric_keys.issubset(final_test)
)
frequency_screening_available = (
    screening.get("status") == "INFERENCE_COMPLETE"
    and screening.get("screening_condition_count") == 146
    and screening.get("excluded_hard_fail_count") == 4
    and len(screening.get("predictions", [])) == 146
    and len(screening.get("frequency_summary", [])) == 15
    and screening.get("SCIENTIFIC_TRAINING_RUN") is False
    and screening.get("FINAL_TEST_RERUN") is False
    and screening.get("TEST_EVALUATION_COUNT") == 1
)
evidence_status = {
    "pretraining_backbone_provenance": evidence["pretraining_backbone_provenance"].get("status", "MISSING"),
    "downstream_convergence": "AVAILABLE" if evidence["downstream_convergence"].get("final_status", {}).get("BASELINE_RUN_VALID") is True else "MISSING",
    "loss_weight_sensitivity": evidence["loss_weight_sensitivity"].get("status", "MISSING"),
    "final_test_metrics": "AVAILABLE" if final_test_available else "MISSING",
    "computational_profile": "AVAILABLE" if evidence["computational_profile"].get("suite") == "comprehensive" else "MISSING",
    "frequency_screening": "AVAILABLE" if frequency_screening_available else "MISSING",
}
expected_evidence_status = {
    "pretraining_backbone_provenance": "PASS",
    "downstream_convergence": "AVAILABLE",
    "loss_weight_sensitivity": "GRID_COMPLETE_VALIDATION_ONLY",
    "final_test_metrics": "AVAILABLE",
    "computational_profile": "AVAILABLE",
    "frequency_screening": "AVAILABLE",
}
assert evidence_status == expected_evidence_status
for section, status in evidence_status.items(): print(section, status)


pretraining_backbone_provenance PASS
downstream_convergence AVAILABLE
loss_weight_sensitivity GRID_COMPLETE_VALIDATION_ONLY
final_test_metrics AVAILABLE
computational_profile AVAILABLE
frequency_screening AVAILABLE


## Manuscript sections

1. Pretraining/backbone provenance
2. Downstream convergence
3. Validation metrics
4. Loss-weight sensitivity
5. Final test metrics — completed exactly once and loaded from the persisted result ledger
6. Computational profile
7. Frequency-screening demonstration

In [2]:
availability = pd.DataFrame([{"section": key, "status": status} for key, status in evidence_status.items()])
display(availability)
final_test_metrics = pd.DataFrame([
    {"target": "efficiency", "MAE": final_test["TEST_EFFICIENCY_MAE"], "RMSE": final_test["TEST_EFFICIENCY_RMSE"], "R2": final_test["TEST_EFFICIENCY_R2"], "MAPE_PERCENT": None},
    {"target": "P_loss", "MAE": final_test["TEST_PLOSS_MAE"], "RMSE": final_test["TEST_PLOSS_RMSE"], "R2": final_test["TEST_PLOSS_R2"], "MAPE_PERCENT": None},
    {"target": "LSP_raw", "MAE": final_test["TEST_LSP_MAE"], "RMSE": final_test["TEST_LSP_RMSE"], "R2": final_test["TEST_LSP_R2"], "MAPE_PERCENT": final_test["TEST_LSP_MAPE_PERCENT"]},
])
frequency_summary = pd.DataFrame(screening["frequency_summary"])[[
    "frequency_hz", "n_samples", "mean_predicted_efficiency",
    "mean_predicted_P_loss", "mean_predicted_LSP", "mean_predicted_LSP_std",
]]
display(final_test_metrics)
display(frequency_summary)
print("TEST_EVALUATION_COUNT =", final_test["TEST_EVALUATION_COUNT"])
print("SCIENTIFIC_TRAINING_RUN = FALSE")
print("FINAL_TEST_RERUN = FALSE")
print("TEST_DATA_ACCESSED = FALSE")
print("FREQUENCY_SCREENING_RERUN = FALSE")
print("MISSING_RESULTS_FABRICATED = FALSE")


,section,status
0,pretraining_backbone_provenance,PASS
1,downstream_convergence,AVAILABLE
2,loss_weight_sensitivity,GRID_COMPLETE_VALIDATION_ONLY
3,final_test_metrics,AVAILABLE
4,computational_profile,AVAILABLE
5,frequency_screening,AVAILABLE


,target,MAE,RMSE,R2,MAPE_PERCENT
0,efficiency,1.145666,1.483375,0.981764,NaN
1,P_loss,0.013693,0.016459,0.970617,NaN
2,LSP_raw,0.027951,0.036366,0.916251,4.788048


,frequency_hz,n_samples,mean_predicted_efficiency,mean_predicted_P_loss,mean_predicted_LSP,mean_predicted_LSP_std
0,1000.0,6,65.515090,0.173008,0.628837,0.029453
1,1250.0,10,61.682512,0.203704,0.615910,0.030907
2,1500.0,10,62.177871,0.197458,0.629272,0.029945
3,1750.0,10,64.655362,0.168443,0.677420,0.030498
4,2000.0,10,63.819211,0.170806,0.607359,0.028466
5,2250.0,10,66.581305,0.147494,0.673465,0.029044
6,2500.0,10,67.964186,0.143924,0.642578,0.030477
7,2750.0,10,66.904665,0.151813,0.604607,0.029257
8,3000.0,10,71.635358,0.120130,0.685379,0.028341
9,3250.0,10,69.177361,0.130359,0.611797,0.029687


TEST_EVALUATION_COUNT = 1
SCIENTIFIC_TRAINING_RUN = FALSE
FINAL_TEST_RERUN = FALSE
TEST_DATA_ACCESSED = FALSE
FREQUENCY_SCREENING_RERUN = FALSE
MISSING_RESULTS_FABRICATED = FALSE
